<a href="https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship-Starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content performance changes with content age

The FlyRank study reports a clear content lifecycle pattern. Health score increases through the early lifecycle and reaches its highest level around 61–90 days, followed by a decline after 270 days. The 271–365 day group has a health score of 14, while the 365+ group rises to 25.1.

The paper gives a cautious interpretation of the 365+ recovery: older content can recover when it is updated well, but the result should not be interpreted as evidence that age itself naturally reverses performance decline.

**Methodology question:**  
Are the observed differences in performance across age groups still present after controlling for factors such as content visibility, client, topic, and whether the content was recently refreshed? Without those controls, the comparison is useful as an observed pattern but does not establish that content age itself causes the performance change.


### Finding 2 — Freshness appears to amplify existing content quality

The paper reports that freshness works differently depending on the underlying quality and depth of the content. In the age × freshness analysis, old content that was recently refreshed had a health score close to that of young and recently refreshed content.

The paper also emphasizes that freshness does not replace quality: refreshing strong content appears more useful than simply refreshing thin or weak content.

**Methodology question:**  
Were refreshed pages compared with a similar group of pages that were not refreshed? If pages were selected for refresh because they already had stronger visibility, strategic importance, or higher quality, selection effects could contribute to the observed improvement. A matched or time-aware before/after comparison would provide stronger evidence for the effect of refreshing.


### Validation relevance

These findings reinforce the need to distinguish between an observed association and a predictive or causal claim. The paper itself treats its ML analysis as exploratory and states that correlations do not prove causation.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 model used Logistic Regression to rank content by its estimated probability of decline.

The important validation question is whether the model performs on clients it has never seen during training. A random row-level split can place pages from the same client in both training and testing, allowing client-specific patterns to be shared across the split.

I therefore compare two evaluation designs:

1. **Before:** a random row-level train/test split.
2. **After:** a client-grouped split where all pages belonging to a client are kept entirely in either training or testing.

The metric is Precision@10, matching the ranking objective used in the Week-5 analysis.

The grouped split is the more deployment-relevant result when the intended use is to rank content for clients that were not represented during model fitting.

### Loading  the dataset

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("/content/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

feature_cols = [
    "ctr",
    "days_since_last_update",
    "word_count",
    "impressions_90d",
    "avg_position",
    "search_volume",
    "competition",
    "cpc"
]

target_col = "is_declining_label"
group_col = "client_id"

X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
y = df[target_col]
groups = df[group_col]

print("Rows:", len(df))
print("Model features:", len(feature_cols))
print("Declining base rate:", round(y.mean(), 3))
print("Unique clients:", groups.nunique())

Rows: 30000
Model features: 8
Declining base rate: 0.542
Unique clients: 32


### Precision@10 helper

In [2]:
def precision_at_k(scores, labels, k=10):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    top_k_labels = labels[order[:k]]

    return top_k_labels.mean()

### Model definition

In [3]:
def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ])

### BEFORE: random split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = make_model()
random_model.fit(X_train, y_train)

random_prob = random_model.predict_proba(X_test)[:, 1]

random_precision_at_10 = precision_at_k(
    random_prob,
    y_test,
    k=10
)

print("Random row-level split")
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Precision@10:", round(random_precision_at_10, 3))
print("Test base rate:", round(y_test.mean(), 3))

Random row-level split
Train rows: 24000
Test rows: 6000
Precision@10: 0.6
Test base rate: 0.542


### AFTER: client-grouped split

In [5]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

grouped_model = make_model()
grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_prob = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_precision_at_10 = precision_at_k(
    grouped_prob,
    y_test_grouped,
    k=10
)

print("Client-grouped split")
print("Train rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))
print("Precision@10:", round(grouped_precision_at_10, 3))
print("Test base rate:", round(y_test_grouped.mean(), 3))

overlap = set(groups_train) & set(groups_test)

print("Clients in both train and test:", len(overlap))

Client-grouped split
Train rows: 23837
Test rows: 6163
Precision@10: 0.4
Test base rate: 0.511
Clients in both train and test: 0


### Before/After comparison

In [6]:
comparison = pd.DataFrame({
    "approach": [
        "Random row-level split",
        "Client-grouped split"
    ],
    "precision_at_10": [
        random_precision_at_10,
        grouped_precision_at_10
    ],
    "test_base_rate": [
        y_test.mean(),
        y_test_grouped.mean()
    ]
})

display(comparison)

,approach,precision_at_10,test_base_rate
0,Random row-level split,0.6,0.542000
1,Client-grouped split,0.4,0.510952


In [7]:
gap = random_precision_at_10 - grouped_precision_at_10

print(
    f"Precision@10 gap (random - grouped): {gap:+.3f}"
)

Precision@10 gap (random - grouped): +0.200


### Interpretation

The random row-level split achieved a Precision@10 of **0.60**, while the client-grouped split achieved **0.40**.

This 0.20-point drop shows that the random split gives a more optimistic estimate of ranking performance. When pages from the same clients can appear in both training and testing, the model can benefit from client-specific patterns that would not necessarily be available for a completely new client.

The client-grouped evaluation is therefore the more honest estimate for the intended deployment scenario. It keeps every client's pages entirely within either the training set or the test set, and the experiment confirmed **0 clients overlap** between the two groups.

The grouped test set also has a lower declining base rate (**0.511**) than the random test set (**0.542**), so the two Precision@10 values should not be interpreted as a perfectly controlled apples-to-apples comparison. Nevertheless, the large difference provides evidence that validation strategy materially affects the measured performance of the model.

The key lesson is:

> **A model's score is only meaningful when the validation split matches the way the model will be used in the real world.**

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
print("FEATURE LEAKAGE AUDIT")
print("=" * 50)

forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "is_declining",
]

print("\n1. Label-derived columns:")
print("Forbidden columns present:",
      [c for c in feature_cols if c in forbidden_columns])

print("\n2. Feature list:")
for feature in feature_cols:
    print(" -", feature)

print("\n3. Product / decision-derived columns:")
decision_keywords = [
    "flag",
    "score",
    "action",
    "reason",
    "label"
]

decision_columns = [
    c for c in df.columns
    if any(word in c.lower() for word in decision_keywords)
]

print("Potential decision-derived columns in dataset:")
print(decision_columns)

print("\nUsed as model features:")
print([
    c for c in feature_cols
    if c in decision_columns
])

FEATURE LEAKAGE AUDIT

1. Label-derived columns:
Forbidden columns present: []

2. Feature list:
 - ctr
 - days_since_last_update
 - word_count
 - impressions_90d
 - avg_position
 - search_volume
 - competition
 - cpc

3. Product / decision-derived columns:
Potential decision-derived columns in dataset:
['is_declining_label']

Used as model features:
[]


In [9]:
# Deliberate leakage test
# IMPORTANT: trend_pct is NOT retained as a model feature.

leaky_features = feature_cols + ["trend_pct"]

X_leaky = df[leaky_features].replace(
    [np.inf, -np.inf],
    np.nan
)

X_train_leaky = X_leaky.iloc[train_idx]
X_test_leaky = X_leaky.iloc[test_idx]

leaky_model = make_model()

leaky_model.fit(
    X_train_leaky,
    y.iloc[train_idx]
)

leaky_prob = leaky_model.predict_proba(
    X_test_leaky
)[:, 1]

leaky_precision_at_10 = precision_at_k(
    leaky_prob,
    y.iloc[test_idx],
    k=10
)

print("Deliberate leakage test")
print("Features:", leaky_features)
print(
    "Precision@10 with trend_pct:",
    round(leaky_precision_at_10, 3)
)
print(
    "Honest grouped Precision@10:",
    round(grouped_precision_at_10, 3)
)

Deliberate leakage test
Features: ['ctr', 'days_since_last_update', 'word_count', 'impressions_90d', 'avg_position', 'search_volume', 'competition', 'cpc', 'trend_pct']
Precision@10 with trend_pct: 1.0
Honest grouped Precision@10: 0.4


In [10]:
# Future / overlapping window audit

print("\nWINDOW AUDIT")
print("=" * 50)

print("Target:")
print("trend_direction / is_declining_label")
print("Derived from the recent 30-day trend relative to the previous 30 days.")

print("\nPotentially overlapping model features:")
overlapping_features = [
    "impressions_90d",
    "ctr",
    "avg_position"
]

for feature in overlapping_features:
    if feature in feature_cols:
        print(f" - {feature}: REVIEW REQUIRED")


WINDOW AUDIT
Target:
trend_direction / is_declining_label
Derived from the recent 30-day trend relative to the previous 30 days.

Potentially overlapping model features:
 - impressions_90d: REVIEW REQUIRED
 - ctr: REVIEW REQUIRED
 - avg_position: REVIEW REQUIRED


### Leakage audit verdict

The model features contain no direct label-derived columns and no existing product decision flags. The deliberate leakage test confirmed the audit is sensitive: adding `trend_pct`, which is related to the target construction, increased Precision@10 from 0.4 to 1.0. This feature is therefore excluded from the model.

However, `impressions_90d`, `ctr`, and `avg_position` require caution because their measurement windows may overlap the period used to define the trend outcome. The current model should therefore be described as measured decision-support on the available snapshot, not as evidence of deployment-ready prospective prediction.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Week-5 Logistic Regression model achieved a higher Precision@10 than the Week-4 baseline, suggesting that the model provides useful predictive signal for identifying declining content.

### Audited claim

On the evaluated snapshot, Logistic Regression achieved a Precision@10 of 0.40 under the client-grouped split, compared with 0.30 for the Week-4 baseline. This is an observed improvement on this evaluation set and suggests that the model may provide useful directional decision-support for prioritizing content.

However, the random row-level split produced a higher Precision@10 of 0.60, demonstrating that the measured score depends materially on the validation design. In addition, several 90-day performance features may overlap with the outcome window used to construct the decline label.

Therefore, the result should not be interpreted as proof of deployment-ready prospective prediction or as evidence that the model causes better content decisions. The current evidence supports an observed, directional result that requires a strictly time-separated feature window and further validation before stronger predictive claims can be made.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.